# Attack Detector

**Research question:** Can Attn-last heatmap *shape* (spiky vs spread) separate typographic-attacked images from clean ones — and does gating `cc_bbox_blur` on that detector cut Clean Δ without giving up attacked accuracy?

**Motivation:** Today `four_lang_cc_bbox_blur` **always** blurs. Earlier `gated_peakiness` never fired. This notebook learns a richer detector.

**Scope:** EN ∩ L for `L ∈ {zh, ko, ja}`, `multi` dual-box attack, frozen `attack_pos` (PROTOCOL).

| Step | Partners | Output |
|------|----------|--------|
| 1 | EN ∩ ZH | `results/zh/multi/` |
| 2 | EN ∩ KO, EN ∩ JA | `results/ko/multi/`, `results/ja/multi/` |

**Phases (per partner):** A Look (PCA/t-SNE) → B Learn (logistic/SVM) → C Use (gated defense).

Protocol: [`../PROTOCOL.md`](../PROTOCOL.md).


In [ ]:
import importlib.util, subprocess, sys

# Only pip-install missing packages — full resolve can take minutes every run.
_REQUIRED = [
    ('open_clip', 'open_clip_torch'),
    ('transformers', 'transformers'),
    ('datasets', 'datasets'),
    ('matplotlib', 'matplotlib'),
    ('PIL', 'Pillow'),
    ('scipy', 'scipy'),
    ('sklearn', 'scikit-learn'),
]
_missing = [pip for mod, pip in _REQUIRED if importlib.util.find_spec(mod) is None]
if _missing:
    print('Installing missing:', _missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)
else:
    print('Deps already installed — skipping pip.')


In [ ]:
import os, platform, random, json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import (
    ChineseCLIPModel, ChineseCLIPProcessor, AutoModel, AutoProcessor,
)
from scipy import ndimage
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.calibration import CalibratedClassifierCV

os.makedirs('results', exist_ok=True)

assert torch.cuda.is_available(), 'CUDA required — install a CUDA build of PyTorch'
DEVICE = 'cuda'
print('Device:', DEVICE, torch.cuda.get_device_name(0))

DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
BLUR_RADIUS  = 12
PARTNER_LANGS = ['zh', 'ko', 'ja']
ATTACK       = 'multi'
DEFENSE_THR  = 0.95  # PROTOCOL floor
SPLIT_SEED   = 0
TRAIN_FRAC   = 0.70
VAL_FRAC     = 0.15
ATTACK_RECALL_TARGET = 0.99  # catch almost all attacks so gated atk drop stays <1pp
# If True, skip partners that already have gated_comparison.json.
SKIP_EXISTING = True  # caches + results from recall=0.99 retune are current

CLASSES = {
    'en': ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'],
    'zh': ['飞机','汽车','鸟','猫','鹿','狗','青蛙','马','船','卡车'],
    'ko': ['비행기','자동차','새','고양이','사슴','개','개구리','말','배','트럭'],
    'ja': ['飛行機','自動車','鳥','猫','鹿','犬','カエル','馬','船','トラック'],
}
TMPL = {
    'en': 'a photo of a {}.',
    'zh': '一张{}的照片。',
    'ko': '{}의 사진.',
    'ja': '{}の写真。',
}
ALL_LANGS = ['en', 'zh', 'ko', 'ja']
print(f'Scope: EN&L for L={PARTNER_LANGS} / {ATTACK} / thr={DEFENSE_THR}')


## Models

EN / ZH / KO / JA CLIP wrappers (PROTOCOL). Defense scores EN ∩ L per partner.


In [ ]:
def _clip_feat(out):
    if torch.is_tensor(out):
        return out
    if getattr(out, 'pooler_output', None) is not None:
        return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    backend = 'open_clip'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms(
            'ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    backend = 'hf_vision'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained(
            'OFA-Sys/chinese-clip-vit-base-patch16',
            attn_implementation='eager').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained(
            'OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True,
                   return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'],
                                       attention_mask=t['attention_mask'],
                                       token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

class KoCLIP:
    lang = 'ko'
    backend = 'hf_vision'
    def __init__(self):
        self.m = AutoModel.from_pretrained(
            'Bingsu/clip-vit-base-patch32-ko',
            attn_implementation='eager').to(DEVICE).eval()
        self.p = AutoProcessor.from_pretrained('Bingsu/clip-vit-base-patch32-ko')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['ko'].format(w) for w in words], padding=True,
                   return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'],
                                       attention_mask=t['attention_mask'])
        return F.normalize(_clip_feat(out), dim=-1)

class JaCLIP:
    lang = 'ja'
    backend = 'open_clip'
    def __init__(self):
        mid = 'hf-hub:llm-jp/llm-jp-clip-vit-base-patch16'
        self.m, _, self.pp = open_clip.create_model_and_transforms(mid)
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer(mid)
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['ja'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

def classify_batch(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

MODEL_CLS = {'en': EnCLIP, 'zh': ZhCLIP, 'ko': KoCLIP, 'ja': JaCLIP}
models = {}
for lang, cls in MODEL_CLS.items():
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in ALL_LANGS}
print('Models ready.')


## Data

CIFAR-10 balanced 1000 + frozen `attack_pos`. Build clean images and `multi` dual-box attacks per partner L.


In [ ]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000 and np.array_equal(true, np.array(_saved['true']))

attack_pos = _saved['attack_pos']
assert len(attack_pos['en']) == len(idx) and len(attack_pos['l']) == len(idx)

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

n_images = len(clean_224)
print(f'Loaded {n_images} images')
print(f"Attack positions: frozen (ref {attack_pos['ref_bw']}x{attack_pos['ref_bh']})")

_FONT_CACHE = {}

def _font_paths():
    if platform.system() == 'Windows':
        wf = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk = os.path.join(wf, 'msyh.ttc')
        lat = os.path.join(wf, 'arial.ttf')
        ko  = os.path.join(wf, 'malgun.ttf')
        if not os.path.isfile(ko):
            ko = cjk
        return cjk, lat, ko
    cjk = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
    lat = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
    if not os.path.isfile(cjk):
        cjk = '/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf'
    return cjk, lat, cjk

_CJK_FONT, _LAT_FONT, _KO_FONT = _font_paths()

def _font_for_lang(lang):
    if lang == 'en':
        return _LAT_FONT
    if lang == 'ko':
        return _KO_FONT
    return _CJK_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:
            _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except Exception:
            _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _clamp_xy(xy, bw, bh):
    x, y = int(xy[0]), int(xy[1])
    x = max(0, min(x, max(0, DISPLAY_SIZE - bw)))
    y = max(0, min(y, max(0, DISPLAY_SIZE - bh)))
    return x, y

def draw_dual_box(img, word0, lang0, word1, lang1, img_idx, already_224=False):
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    xy0 = attack_pos['en'][int(img_idx)]
    xy1 = attack_pos['l'][int(img_idx)]
    for word, lang, xy in [(word0, lang0, xy0), (word1, lang1, xy1)]:
        font = _get_font(_font_for_lang(lang))
        bb   = draw.textbbox((0, 0), word, font=font)
        bw   = (bb[2] - bb[0]) + 2 * PAD
        bh   = (bb[3] - bb[1]) + PAD + 12
        rx, ry = _clamp_xy(xy, bw, bh)
        draw.rectangle([rx, ry, rx + bw, ry + bh], fill='white')
        draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)
    return img

def build_attack_multi(L):
    out = []
    for i in range(n_images):
        t = int(target[i])
        out.append(draw_dual_box(
            clean_224[i], CLASSES['en'][t], 'en', CLASSES[L][t], L, i, True))
    return out

attacked_by_L = {L: build_attack_multi(L) for L in PARTNER_LANGS}
print('Built multi attacks for:', list(attacked_by_L.keys()))

_strip = Image.new('RGB', (DISPLAY_SIZE * 4, 80), (240, 240, 240))
_sd = ImageDraw.Draw(_strip)
for j, (lang, word) in enumerate([
    ('en', 'airplane'), ('zh', '飞机'), ('ko', '비행기'), ('ja', '飛行機')
]):
    f = _get_font(_font_for_lang(lang), 28)
    _sd.text((j * DISPLAY_SIZE + 10, 20), f'{lang}: {word}', fill='black', font=f)
_strip.save('results/font_check.png')
print('Fonts: LAT=', _LAT_FONT, 'CJK=', _CJK_FONT, 'KO=', _KO_FONT)

clean_acc = {
    ml: float((classify_batch(models[ml], clean_224, CLASSES[ml]) == true).mean())
    for ml in ALL_LANGS
}
print('Clean acc:', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})


## Attn-last saliency

CLS→patch attention from the last ViT block (same as four_lang).


In [ ]:
def _norm_cam(cam):
    cam = np.maximum(cam if isinstance(cam, np.ndarray) else cam.cpu().numpy(), 0)
    cam -= cam.min()
    mx = cam.max()
    return cam / mx if mx > 0 else cam

def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ) / 255.0

def _make_openclip_hook(collector):
    def hook(module, inputs, output):
        q_in = inputs[0]
        if getattr(module, 'batch_first', False):
            B, L, D = q_in.shape
        else:
            L, B, D = q_in.shape
            q_in = q_in.transpose(0, 1).contiguous()
        n_head = module.num_heads
        hd = D // n_head
        with torch.no_grad():
            qkv = F.linear(q_in, module.in_proj_weight, module.in_proj_bias)
            q, k, _ = qkv.chunk(3, dim=-1)
            q = q.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            k = k.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            attn = (q @ k.transpose(-2, -1)) * (hd ** -0.5)
            attn = attn.softmax(dim=-1)
        collector.append(attn[0].detach().cpu())
    return hook

def _build_attn_cam(all_attns, variant='last'):
    a = all_attns[-1]
    cls_row = a.mean(0)[0, 1:]
    if variant == 'rollout':
        L = all_attns[0].shape[-1]
        rollout = torch.eye(L)
        for att in all_attns:
            a_r = 0.5 * att.mean(0) + 0.5 * torch.eye(L)
            a_r = a_r / a_r.sum(-1, keepdim=True)
            rollout = a_r @ rollout
        cls_row = rollout[0, 1:]
    n = int(round(cls_row.shape[0] ** 0.5))
    return _norm_cam(cls_row.reshape(n, n).numpy())

def classify_and_attn(lang, pil_img, variant='last'):
    wrapper = models[lang]
    if wrapper.backend == 'open_clip':
        x = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
        collector = []
        handles = [
            rb.attn.register_forward_hook(_make_openclip_hook(collector))
            for rb in wrapper.m.visual.transformer.resblocks
        ]
        with torch.no_grad():
            feat = wrapper.m.visual(x)
            imf = F.normalize(feat, dim=-1)
            pred = int((imf @ TEXT_EMB[lang].T).squeeze().argmax().item())
        for h in handles:
            h.remove()
        return pred, _build_attn_cam(collector, variant)

    pv = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        vis_out = wrapper.m.vision_model(pixel_values=pv, output_attentions=True)
        if hasattr(wrapper.m, 'visual_projection'):
            proj = wrapper.m.visual_projection(vis_out.pooler_output)
        else:
            proj = vis_out.pooler_output
        imf = F.normalize(proj, dim=-1)
        pred = int((imf @ TEXT_EMB[lang].T).squeeze().argmax().item())
    attns = [a[0].cpu() for a in vis_out.attentions]
    return pred, _build_attn_cam(attns, variant)

for lang in ALL_LANGS:
    p, cam = classify_and_attn(lang, clean_224[0], 'last')
    print(f'  {lang}: pred={p} cam={cam.shape}')
print('Saliency ready.')


## Mask helpers

`cc_bbox_blur`: intersection → percentile thr → dilate → top-2 CC + bbox → Gaussian blur fill.


In [ ]:
def n_cam_intersection(*cams):
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2, :-2] | pad[:-2, 1:-1] | pad[:-2, 2:] |
             pad[1:-1, :-2] | pad[1:-1, 1:-1] | pad[1:-1, 2:] |
             pad[2:, :-2] | pad[2:, 1:-1] | pad[2:, 2:])
    return m

def cam_to_mask(saliency, threshold=0.85, dilate=3):
    thr = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0:
        mask = dilate_mask(mask, iterations=dilate)
    return mask

def filter_mask_components(mask, top_k=2, bbox_snap=False):
    labeled, n = ndimage.label(mask.astype(bool))
    if n == 0:
        return mask.astype(bool)
    sizes = [(labeled == i).sum() for i in range(1, n + 1)]
    keep = set(np.argsort(sizes)[::-1][:top_k] + 1)
    out = np.zeros_like(mask, dtype=bool)
    for i in keep:
        comp = labeled == i
        if bbox_snap:
            ys, xs = np.where(comp)
            out[ys.min():ys.max() + 1, xs.min():xs.max() + 1] = True
        else:
            out |= comp
    return out

def apply_mask(pil_img, mask, fill='blur'):
    arr = np.array(pil_img.convert('RGB'))
    m = mask.astype(bool)
    if mask.shape != arr.shape[:2]:
        m = np.array(Image.fromarray(m.astype(np.uint8) * 255).resize(
            arr.shape[1::-1], Image.NEAREST)) > 127
    out = arr.copy()
    if fill == 'blur':
        blurred = np.array(Image.fromarray(arr).filter(
            ImageFilter.GaussianBlur(radius=BLUR_RADIUS)))
        out[m] = blurred[m]
    else:
        mean = arr[~m].mean(0) if (~m).any() else arr.reshape(-1, 3).mean(0)
        out[m] = mean
    return Image.fromarray(out.astype(np.uint8))

def build_cc_bbox_blur_mask(cam_en, cam_l, threshold=0.95, dilate=3, top_k=2):
    inter = n_cam_intersection(cam_en, cam_l)
    mask = cam_to_mask(inter, threshold, dilate=dilate)
    return filter_mask_components(mask, top_k=top_k, bbox_snap=True)

print('Masking helpers ready.')


## Per-partner pipeline

Bake Attn-last → features → PCA/t-SNE → detector → gated `cc_bbox_blur`. Writes `results/{L}/multi/`.


In [ ]:
def load_or_bake_pair(L, attacked_imgs, out_dir):
    cache_dir = Path(out_dir) / 'cache'
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f'attn_en_{L}_clean_multi.npz'
    if cache_path.exists():
        print(f'Loading cache {cache_path}')
        z = np.load(cache_path, allow_pickle=False)
        # Support legacy zh key names from Step 1
        def _get(prefix, new, old):
            if new in z.files:
                return z[new]
            return z[old]
        return {
            'clean': {
                'cams_en': z['clean_cams_en'],
                'cams_l': _get('clean', 'clean_cams_l', 'clean_cams_zh'),
                'preds_en': z['clean_preds_en'],
                'preds_l': _get('clean', 'clean_preds_l', 'clean_preds_zh'),
            },
            'atk': {
                'cams_en': z['atk_cams_en'],
                'cams_l': _get('atk', 'atk_cams_l', 'atk_cams_zh'),
                'preds_en': z['atk_preds_en'],
                'preds_l': _get('atk', 'atk_preds_l', 'atk_preds_zh'),
            },
        }

    def _bake(images, tag):
        ce, cl, pe, pl = [], [], [], []
        t0 = time.time()
        for i, img in enumerate(images):
            if i % 100 == 0:
                print(f'  [{tag}] {i}/{len(images)}...', flush=True)
            p_en, c_en = classify_and_attn('en', img, 'last')
            p_l, c_l = classify_and_attn(L, img, 'last')
            pe.append(p_en); ce.append(c_en.astype(np.float32))
            pl.append(p_l); cl.append(c_l.astype(np.float32))
        print(f'  [{tag}] done in {time.time()-t0:.1f}s')
        return {
            'cams_en': np.stack(ce), 'cams_l': np.stack(cl),
            'preds_en': np.array(pe, dtype=np.int64),
            'preds_l': np.array(pl, dtype=np.int64),
        }

    print(f'Baking clean saliency EN&{L}...')
    clean = _bake(clean_224, f'clean/{L}')
    print(f'Baking attacked saliency EN&{L}...')
    atk = _bake(attacked_imgs, f'atk/{L}')
    np.savez_compressed(
        cache_path,
        clean_cams_en=clean['cams_en'], clean_cams_l=clean['cams_l'],
        clean_preds_en=clean['preds_en'], clean_preds_l=clean['preds_l'],
        atk_cams_en=atk['cams_en'], atk_cams_l=atk['cams_l'],
        atk_preds_en=atk['preds_en'], atk_preds_l=atk['preds_l'],
    )
    print(f'Saved {cache_path}')
    return {'clean': clean, 'atk': atk}

print('Bake helpers ready.')


## Feature extraction helpers


In [ ]:
def _entropy(p):
    p = p.ravel().astype(np.float64)
    p = p - p.min()
    s = p.sum()
    if s <= 0:
        return 0.0
    p = p / s
    p = p[p > 0]
    return float(-(p * np.log(p + 1e-12)).sum())

def _topk_mass(p, frac):
    flat = p.ravel().astype(np.float64)
    flat = flat - flat.min()
    s = flat.sum()
    if s <= 0:
        return 0.0
    k = max(1, int(round(len(flat) * frac)))
    top = np.partition(flat, -k)[-k:]
    return float(top.sum() / s)

def _gini(p):
    flat = np.sort(p.ravel().astype(np.float64))
    n = len(flat)
    if n == 0 or flat.sum() <= 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return float((2 * (idx * flat).sum()) / (n * flat.sum()) - (n + 1) / n)

def _spatial_kurtosis(p):
    flat = p.ravel().astype(np.float64)
    mu = flat.mean()
    sig = flat.std()
    if sig < 1e-12:
        return 0.0
    z = (flat - mu) / sig
    return float((z ** 4).mean() - 3.0)

def _map_features(cam, prefix):
    a = align_cam(cam)
    mx = float(a.max())
    mn = float(a.mean()) + 1e-12
    cov95 = float((a >= np.percentile(a, 95)).mean())
    _, ncc = ndimage.label(a >= np.percentile(a, 95))
    return {
        f'{prefix}_entropy': _entropy(a),
        f'{prefix}_topk05': _topk_mass(a, 0.05),
        f'{prefix}_topk10': _topk_mass(a, 0.10),
        f'{prefix}_max_over_mean': mx / mn,
        f'{prefix}_gini': _gini(a),
        f'{prefix}_kurtosis': _spatial_kurtosis(a),
        f'{prefix}_cov95': cov95,
        f'{prefix}_ncc95': float(ncc),
    }

def extract_pair_features(cam_en, cam_l):
    feats = {}
    feats.update(_map_features(cam_en, 'en'))
    feats.update(_map_features(cam_l, 'l'))
    inter = n_cam_intersection(cam_en, cam_l)
    feats.update(_map_features(inter, 'inter'))
    ae, al = align_cam(cam_en), align_cam(cam_l)
    feats['en_l_corr'] = float(np.corrcoef(ae.ravel(), al.ravel())[0, 1]) if ae.std() > 0 and al.std() > 0 else 0.0
    hot_e = ae >= np.percentile(ae, 95)
    hot_l = al >= np.percentile(al, 95)
    union = (hot_e | hot_l).sum()
    feats['en_l_iou95'] = float((hot_e & hot_l).sum() / union) if union > 0 else 0.0
    return feats

def build_feature_matrix(clean_cams, atk_cams):
    rows_feat, y_labels, img_ids = [], [], []
    for i in range(n_images):
        rows_feat.append(extract_pair_features(clean_cams['cams_en'][i], clean_cams['cams_l'][i]))
        y_labels.append(0); img_ids.append(i)
    for i in range(n_images):
        rows_feat.append(extract_pair_features(atk_cams['cams_en'][i], atk_cams['cams_l'][i]))
        y_labels.append(1); img_ids.append(i)
    names = list(rows_feat[0].keys())
    X = np.array([[r[k] for k in names] for r in rows_feat], dtype=np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y = np.array(y_labels, dtype=np.int64)
    img_ids = np.array(img_ids, dtype=np.int64)
    return X, y, img_ids, names

print('Feature helpers ready.')


## Run partners (Step 1 ZH, Step 2 KO+JA)


In [ ]:
class _ScaledModel:
    def __init__(self, scaler, model):
        self.scaler = scaler
        self.model = model
    def predict_proba(self, X_):
        return self.model.predict_proba(self.scaler.transform(X_))
    def predict(self, X_):
        return self.model.predict(self.scaler.transform(X_))

def defend_images(images, cams_en, cams_l, gate_flags, threshold=DEFENSE_THR, label=''):
    t0 = time.time()
    out, covs, n_defended = [], [], 0
    for i, img in enumerate(images):
        if gate_flags is not None and not gate_flags[i]:
            out.append(img); covs.append(0.0); continue
        mask = build_cc_bbox_blur_mask(cams_en[i], cams_l[i], threshold=threshold)
        covs.append(float(mask.mean()))
        out.append(apply_mask(img, mask, fill='blur'))
        n_defended += 1
    print(f'  [{label}] defended={n_defended}/{len(images)}  '
          f'cov={100*np.mean(covs):.1f}%  {time.time()-t0:.1f}s')
    return out, float(np.mean(covs)), n_defended / len(images)

def score_pair(imgs, L, label=''):
    preds = {
        'en': classify_batch(models['en'], imgs, CLASSES['en']),
        L: classify_batch(models[L], imgs, CLASSES[L]),
    }
    acc = {ml: float((preds[ml] == true).mean()) for ml in ['en', L]}
    asr = {ml: float((preds[ml] == target).mean()) for ml in ['en', L]}
    print(f'  [{label}] acc EN={100*acc["en"]:.1f}% {L.upper()}={100*acc[L]:.1f}%  '
          f'ASR EN={100*asr["en"]:.1f}% {L.upper()}={100*asr[L]:.1f}%')
    return preds, acc, asr

def run_partner(L):
    out_dir = Path('results') / L / ATTACK
    out_dir.mkdir(parents=True, exist_ok=True)
    gated_path = out_dir / 'gated_comparison.json'
    if SKIP_EXISTING and gated_path.exists():
        print(f'\n===== SKIP {L} (existing {gated_path}) =====')
        with open(gated_path, encoding='utf-8') as f:
            return json.load(f)

    print(f'\n===== RUN EN&{L} / {ATTACK} =====')
    attacked = attacked_by_L[L]
    packed = load_or_bake_pair(L, attacked, out_dir)
    clean_cams, atk_cams = packed['clean'], packed['atk']
    print(f'Clean pred acc EN/{L}:',
          float((clean_cams['preds_en'] == true).mean()),
          float((clean_cams['preds_l'] == true).mean()))
    print(f'Atk   pred acc EN/{L}:',
          float((atk_cams['preds_en'] == true).mean()),
          float((atk_cams['preds_l'] == true).mean()))

    X, y, img_ids, feature_names = build_feature_matrix(clean_cams, atk_cams)
    print(f'Features: {X.shape}')

    rng_split = np.random.RandomState(SPLIT_SEED)
    perm = rng_split.permutation(n_images)
    n_train = int(round(TRAIN_FRAC * n_images))
    n_val = int(round(VAL_FRAC * n_images))
    train_imgs = set(perm[:n_train].tolist())
    val_imgs = set(perm[n_train:n_train + n_val].tolist())
    test_imgs = set(perm[n_train + n_val:].tolist())
    train_m = np.array([i in train_imgs for i in img_ids])
    val_m = np.array([i in val_imgs for i in img_ids])
    test_m = np.array([i in test_imgs for i in img_ids])

    # --- Phase A ---
    scaler_viz = StandardScaler().fit(X[train_m])
    Xz = scaler_viz.transform(X)
    pca = PCA(n_components=2, random_state=SPLIT_SEED)
    Xp = pca.fit_transform(Xz)
    c0 = Xp[train_m & (y == 0)].mean(0)
    c1 = Xp[train_m & (y == 1)].mean(0)
    pca_pred = ((Xp - c0) ** 2).sum(1) > ((Xp - c1) ** 2).sum(1)
    pca_acc_test = float((pca_pred[test_m] == y[test_m]).mean())
    fig, ax = plt.subplots(figsize=(6, 5))
    for lab, name, color in [(0, 'clean', '#4C78A8'), (1, 'attacked', '#E45756')]:
        m = y == lab
        ax.scatter(Xp[m, 0], Xp[m, 1], s=8, alpha=0.45, c=color, label=name, edgecolors='none')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title(f'PCA EN&{L} (test NN-cent={100*pca_acc_test:.1f}%)')
    ax.legend(frameon=False); fig.tight_layout()
    fig.savefig(out_dir / 'pca_features.png', dpi=150); plt.close(fig)

    print('Running t-SNE...')
    Xt = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca',
              random_state=SPLIT_SEED).fit_transform(Xz)
    fig, ax = plt.subplots(figsize=(6, 5))
    for lab, name, color in [(0, 'clean', '#4C78A8'), (1, 'attacked', '#E45756')]:
        m = y == lab
        ax.scatter(Xt[m, 0], Xt[m, 1], s=8, alpha=0.45, c=color, label=name, edgecolors='none')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.set_title(f't-SNE EN&{L} heatmap features')
    ax.legend(frameon=False); fig.tight_layout()
    fig.savefig(out_dir / 'tsne_features.png', dpi=150); plt.close(fig)

    phase_a = {
        'pca_explained_variance_sum': float(pca.explained_variance_ratio_.sum()),
        'pca_nearest_centroid_test_acc': pca_acc_test,
        'n_features': len(feature_names),
    }
    with open(out_dir / 'phase_a_summary.json', 'w', encoding='utf-8') as f:
        json.dump(phase_a, f, indent=2)

    # --- Phase B ---
    X_tr, y_tr = X[train_m], y[train_m]
    X_va, y_va = X[val_m], y[val_m]
    X_te, y_te = X[test_m], y[test_m]

    logit = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SPLIT_SEED)),
    ])
    logit.fit(X_tr, y_tr)

    _svm_scaler = StandardScaler().fit(X_tr)
    svm_inner = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', max_iter=5000, random_state=SPLIT_SEED), cv=3)
    svm_inner.fit(_svm_scaler.transform(X_tr), y_tr)
    svm = _ScaledModel(_svm_scaler, svm_inner)

    def _eval(model, X_, y_, name):
        proba = model.predict_proba(X_)[:, 1]
        pred = (proba >= 0.5).astype(int)
        auc = float(roc_auc_score(y_, proba))
        acc = float(accuracy_score(y_, pred))
        p, r, f1, _ = precision_recall_fscore_support(y_, pred, average='binary', zero_division=0)
        print(f'  {name}: acc={100*acc:.1f}% AUC={auc:.3f} P={p:.3f} R={r:.3f}')
        return {'acc': acc, 'auc': auc, 'precision': float(p), 'recall': float(r),
                'f1': float(f1), 'proba': proba}

    logit_va, logit_te = _eval(logit, X_va, y_va, 'logit/val'), _eval(logit, X_te, y_te, 'logit/test')
    svm_va, svm_te = _eval(svm, X_va, y_va, 'svm/val'), _eval(svm, X_te, y_te, 'svm/test')
    if logit_va['auc'] >= svm_va['auc']:
        primary_name, primary, primary_va, primary_te = 'logistic', logit, logit_va, logit_te
    else:
        primary_name, primary, primary_va, primary_te = 'linear_svm', svm, svm_va, svm_te
    print(f'  Primary: {primary_name}')

    proba_va = primary_va['proba']
    _, tpr, thr_curve = roc_curve(y_va, proba_va)
    candidates = [(float(th), float(rec)) for th, rec in zip(thr_curve, tpr) if not np.isnan(th)]
    ok = [c for c in candidates if c[1] >= ATTACK_RECALL_TARGET]
    if ok:
        best_thr = max(ok, key=lambda c: c[0])[0]
    else:
        best_f1, best_thr = -1.0, 0.5
        for th in np.linspace(0.05, 0.95, 37):
            pred = (proba_va >= th).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(y_va, pred, average='binary', zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, float(th)
    print(f'  threshold={best_thr:.4f}')

    def metrics_at_thr(proba, y_, thr):
        pred = (proba >= thr).astype(int)
        auc = float(roc_auc_score(y_, proba))
        acc = float(accuracy_score(y_, pred))
        p, r, f1, _ = precision_recall_fscore_support(y_, pred, average='binary', zero_division=0)
        return {
            'acc': acc, 'auc': auc, 'precision': float(p), 'recall': float(r), 'f1': float(f1),
            'confusion_matrix': confusion_matrix(y_, pred).tolist(),
            'fire_rate_clean': float(pred[y_ == 0].mean()) if (y_ == 0).any() else 0.0,
            'fire_rate_attacked': float(pred[y_ == 1].mean()) if (y_ == 1).any() else 0.0,
            'threshold': thr,
        }

    te_at_thr = metrics_at_thr(primary_te['proba'], y_te, best_thr)
    va_at_thr = metrics_at_thr(proba_va, y_va, best_thr)
    print('  Test@thr:', {k: te_at_thr[k] for k in
          ('acc', 'auc', 'recall', 'fire_rate_clean', 'fire_rate_attacked')})

    weights = logit.named_steps['clf'].coef_.ravel()
    order = np.argsort(np.abs(weights))[::-1]
    fig, ax = plt.subplots(figsize=(7, 6))
    top_k = min(20, len(feature_names))
    ax.barh(range(top_k), weights[order[:top_k]][::-1], color='#72B7B2')
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([feature_names[i] for i in order[:top_k]][::-1], fontsize=8)
    ax.set_xlabel('Logistic coefficient')
    ax.set_title(f'Feature importance EN&{L}')
    fig.tight_layout(); fig.savefig(out_dir / 'feature_importance.png', dpi=150); plt.close(fig)

    all_proba = primary.predict_proba(X)[:, 1]
    all_pred = (all_proba >= best_thr).astype(int)
    gate_clean = all_pred[:n_images].astype(bool)
    gate_attacked = all_pred[n_images:].astype(bool)
    print(f'  Gate fire clean={100*gate_clean.mean():.1f}% attacked={100*gate_attacked.mean():.1f}%')

    detector_metrics = {
        'L': L, 'attack': ATTACK, 'primary': primary_name, 'threshold': best_thr,
        'attack_recall_target': ATTACK_RECALL_TARGET,
        'feature_names': feature_names,
        'logistic_val_auc_thr05': logit_va['auc'],
        'logistic_test_auc_thr05': logit_te['auc'],
        'svm_val_auc_thr05': svm_va['auc'],
        'svm_test_auc_thr05': svm_te['auc'],
        'val_at_threshold': {k: v for k, v in va_at_thr.items()},
        'test_at_threshold': {k: v for k, v in te_at_thr.items()},
        'logistic_top_weights': [
            {'feature': feature_names[i], 'weight': float(weights[i])}
            for i in order[:15].tolist()
        ],
        'phase_a_pca_nn_cent_acc': pca_acc_test,
    }
    with open(out_dir / 'detector_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(detector_metrics, f, indent=2)

    # --- Phase C ---
    policies = {}
    print('=== never_defend / attacked ===')
    _, base_acc, base_asr = score_pair(attacked, L, 'atk/never')
    policies['never_defend'] = {
        'attacked_acc': base_acc, 'attacked_asr': base_asr,
        'clean_acc': {ml: clean_acc[ml] for ml in ['en', L]},
        'clean_delta': {ml: 0.0 for ml in ['en', L]},
        'defend_frac_attacked': 0.0, 'defend_frac_clean': 0.0,
    }

    print('=== always_defend ===')
    always_flags = np.ones(n_images, dtype=bool)
    atk_always, cov_aa, frac_aa = defend_images(
        attacked, atk_cams['cams_en'], atk_cams['cams_l'], always_flags, label='atk/always')
    _, acc_aa, asr_aa = score_pair(atk_always, L, 'atk/always')
    cln_always, cov_ac, frac_ac = defend_images(
        clean_224, clean_cams['cams_en'], clean_cams['cams_l'], always_flags, label='cln/always')
    _, acc_ac, _ = score_pair(cln_always, L, 'cln/always')
    policies['always_defend'] = {
        'attacked_acc': acc_aa, 'attacked_asr': asr_aa,
        'clean_acc_masked': acc_ac,
        'clean_acc': {ml: clean_acc[ml] for ml in ['en', L]},
        'clean_delta': {ml: acc_ac[ml] - clean_acc[ml] for ml in ['en', L]},
        'defend_frac_attacked': frac_aa, 'defend_frac_clean': frac_ac,
        'coverage_attacked': cov_aa, 'coverage_clean': cov_ac,
    }

    print('=== gated ===')
    atk_gated, cov_ga, frac_ga = defend_images(
        attacked, atk_cams['cams_en'], atk_cams['cams_l'], gate_attacked, label='atk/gated')
    _, acc_ga, asr_ga = score_pair(atk_gated, L, 'atk/gated')
    cln_gated, cov_gc, frac_gc = defend_images(
        clean_224, clean_cams['cams_en'], clean_cams['cams_l'], gate_clean, label='cln/gated')
    _, acc_gc, _ = score_pair(cln_gated, L, 'cln/gated')
    policies['gated'] = {
        'attacked_acc': acc_ga, 'attacked_asr': asr_ga,
        'clean_acc_masked': acc_gc,
        'clean_acc': {ml: clean_acc[ml] for ml in ['en', L]},
        'clean_delta': {ml: acc_gc[ml] - clean_acc[ml] for ml in ['en', L]},
        'defend_frac_attacked': frac_ga, 'defend_frac_clean': frac_gc,
        'coverage_attacked': cov_ga, 'coverage_clean': cov_gc,
    }

    def _mean_acc(d):
        return 0.5 * (d['en'] + d[L])
    def _mean_delta(d):
        return 0.5 * (d['en'] + d[L])

    always_mean_acc = _mean_acc(policies['always_defend']['attacked_acc'])
    gated_mean_acc = _mean_acc(policies['gated']['attacked_acc'])
    always_mean_delta = _mean_delta(policies['always_defend']['clean_delta'])
    gated_mean_delta = _mean_delta(policies['gated']['clean_delta'])
    success = {
        'clean_delta_improved_pp': 100 * (gated_mean_delta - always_mean_delta),
        'attacked_acc_drop_pp': 100 * (always_mean_acc - gated_mean_acc),
        'meets_success_bar': (
            (gated_mean_delta - always_mean_delta) >= 0.01
            and (always_mean_acc - gated_mean_acc) <= 0.01
        ),
    }
    print('Success check:', success)

    gated_comparison = {
        'L': L, 'attack': ATTACK, 'defense_threshold': DEFENSE_THR,
        'detector': {
            'primary': primary_name, 'threshold': best_thr,
            'test_auc': te_at_thr['auc'], 'test_recall': te_at_thr['recall'],
            'test_fire_rate_clean': te_at_thr['fire_rate_clean'],
            'test_fire_rate_attacked': te_at_thr['fire_rate_attacked'],
        },
        'policies': policies,
        'summary': {
            'always_mean_attacked_acc': always_mean_acc,
            'gated_mean_attacked_acc': gated_mean_acc,
            'always_mean_clean_delta': always_mean_delta,
            'gated_mean_clean_delta': gated_mean_delta,
            'never_mean_attacked_acc': _mean_acc(base_acc),
        },
        'success': success,
    }
    with open(gated_path, 'w', encoding='utf-8') as f:
        json.dump(gated_comparison, f, indent=2)

    # summary plot
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    names = ['never', 'always', 'gated']
    atk_means = [
        100 * gated_comparison['summary']['never_mean_attacked_acc'],
        100 * always_mean_acc, 100 * gated_mean_acc,
    ]
    colors = ['#E45756', '#4C78A8', '#54A24B']
    axes[0].bar(names, atk_means, color=colors)
    axes[0].set_ylabel('Mean acc EN+L (%)'); axes[0].set_title('Attacked accuracy')
    axes[0].set_ylim(0, 100)
    for i, v in enumerate(atk_means):
        axes[0].text(i, v + 1.5, f'{v:.1f}', ha='center', fontsize=9)
    deltas = [0.0, 100 * always_mean_delta, 100 * gated_mean_delta]
    axes[1].bar(names, deltas, color=colors)
    axes[1].axhline(0, color='gray', lw=0.8)
    axes[1].set_ylabel('Mean Clean delta (pp)'); axes[1].set_title('Clean-image degradation')
    for i, v in enumerate(deltas):
        axes[1].text(i, v - 0.4 if v < 0 else v + 0.3, f'{v:.2f}', ha='center', fontsize=9)
    fig.suptitle(
        f'Gated cc_bbox_blur (EN&{L} / {ATTACK}) | {primary_name} AUC={te_at_thr["auc"]:.3f}',
        fontsize=11)
    fig.tight_layout(); fig.savefig(out_dir / 'gated_comparison.png', dpi=150); plt.close(fig)
    print(f'Saved results under {out_dir}')
    return gated_comparison

print('run_partner ready.')


## Main loop + summary


In [ ]:
all_results = {}
for L in PARTNER_LANGS:
    all_results[L] = run_partner(L)

# Roll-up summary
rows = []
for L, g in all_results.items():
    s = g['summary']
    d = g['detector']
    suc = g['success']
    rows.append({
        'L': L,
        'attack': ATTACK,
        'detector_primary': d['primary'],
        'test_auc': d['test_auc'],
        'test_fire_clean': d['test_fire_rate_clean'],
        'test_fire_attacked': d['test_fire_rate_attacked'],
        'never_mean_atk_acc': s['never_mean_attacked_acc'],
        'always_mean_atk_acc': s['always_mean_attacked_acc'],
        'gated_mean_atk_acc': s['gated_mean_attacked_acc'],
        'always_mean_clean_delta': s['always_mean_clean_delta'],
        'gated_mean_clean_delta': s['gated_mean_clean_delta'],
        'atk_acc_drop_pp': suc['attacked_acc_drop_pp'],
        'clean_delta_improved_pp': suc['clean_delta_improved_pp'],
        'meets_success_bar': suc['meets_success_bar'],
    })

comparison_summary = {
    'attack': ATTACK,
    'defense_threshold': DEFENSE_THR,
    'partners': rows,
}
with open('results/comparison_summary.json', 'w', encoding='utf-8') as f:
    json.dump(comparison_summary, f, indent=2)
print('Saved results/comparison_summary.json')

print('\n=== ROLL-UP ===')
print(f'{"L":<4} {"AUC":>6} {"fireC":>6} {"fireA":>6} {"always":>7} {"gated":>7} '
      f'{"dAtk":>6} {"aΔ":>7} {"gΔ":>7} {"ok":>4}')
for r in rows:
    print(f'{r["L"]:<4} {r["test_auc"]:6.3f} {100*r["test_fire_clean"]:5.1f}% '
          f'{100*r["test_fire_attacked"]:5.1f}% '
          f'{100*r["always_mean_atk_acc"]:6.1f}% {100*r["gated_mean_atk_acc"]:6.1f}% '
          f'{r["atk_acc_drop_pp"]:5.2f} '
          f'{100*r["always_mean_clean_delta"]:6.2f} {100*r["gated_mean_clean_delta"]:6.2f} '
          f'{str(r["meets_success_bar"]):>4}')
